In [1]:
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

train_data = pd.read_csv('../TEST_TRAIN/train.csv')

In [2]:
need_cols2 = ['election','winner', 'country/region','previous_majority_proportion',
       'previous_winner','Conservative', 'Labour', 'LD', 'incumbent',
       'previous_con_share', 'previous_lib_share', 'previous_lab_share',
       'previous_natSW_share', 'projected_con_share', 'projected_lib_share', 'projected_lab_share'
]

train_data2 = train_data[need_cols2]
train_data2 = train_data2[train_data2['winner'] != 'oth'].copy()

In [4]:
election_years = [2005, 2010, 2015, 2017, 2019]
feature_columns2 = need_cols2[2:]  # Exclude 'election', and 'winner'

xgboost_param_grid = {
    'n_estimators': [20,35,50, 100],
    'max_depth': [2, 3, 4, 5]
}

results = []

for election_year in election_years:
    test = train_data2[train_data2['election'] == election_year]
    previous_elections = train_data2[train_data2['election'] < election_year]

    changed_seats = test['previous_winner'] != test['winner']

    previous_winner_predictions = test['previous_winner']
    previous_winner_accuracy = accuracy_score(test['winner'], previous_winner_predictions)

    X_train = pd.get_dummies(previous_elections[feature_columns2])
    y_train = previous_elections['winner']
    X_test = pd.get_dummies(test[feature_columns2]).reindex(columns=X_train.columns, fill_value=0)

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    xgboost = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        n_jobs=-1,
        random_state=42
    )
    xgboost_grid_search = GridSearchCV(
        estimator=xgboost,
        param_grid=xgboost_param_grid,
        scoring='accuracy',
        n_jobs=-1
    )
    xgboost_grid_search.fit(X_train, y_train_encoded)

    xgboost_predictions_encoded = xgboost_grid_search.predict(X_test)
    xgboost_predictions = label_encoder.inverse_transform(xgboost_predictions_encoded)

    xgboost_accuracy = accuracy_score(test['winner'], xgboost_predictions)
    xgboost_changed_accuracy = accuracy_score(
        test.loc[changed_seats, 'winner'],
        xgboost_predictions[changed_seats]
    )

    results.append({
        'election': election_year,
        'previous_winner_accuracy': previous_winner_accuracy,
        'xgboost_accuracy': xgboost_accuracy,
        'xgboost_changed_seats_accuracy': xgboost_changed_accuracy,
        'xgboost_best_cv_accuracy': xgboost_grid_search.best_score_,
        'xgboost_n_estimators': xgboost_grid_search.best_params_['n_estimators'],
        'xgboost_max_depth': xgboost_grid_search.best_params_['max_depth'],
        'changed_seats': changed_seats.sum()
    })

results = pd.DataFrame(results)
accuracy_columns = [
    'previous_winner_accuracy',
    'xgboost_accuracy',
    'xgboost_changed_seats_accuracy',
    'xgboost_best_cv_accuracy'
]

results[accuracy_columns] = results[accuracy_columns].round(3)
results

,election,previous_winner_accuracy,xgboost_accuracy,xgboost_changed_seats_accuracy,xgboost_best_cv_accuracy,xgboost_n_estimators,xgboost_max_depth,changed_seats
0,2005,0.912,0.907,0.018,0.961,35,5,55
1,2010,0.825,0.803,0.127,0.943,20,2,110
2,2015,0.830,0.775,0.168,0.924,20,4,107
3,2017,0.894,0.889,0.060,0.899,20,2,67
4,2019,0.881,0.876,0.040,0.897,20,2,75
